In [ ]:
# In[1]:

import pandas as pd

# Load CSV files (they are in the current working directory)
df_metric = pd.read_csv("metric.csv")
df_trace = pd.read_csv("trace.csv")
df_log = pd.read_csv("log.csv")
df_error = pd.read_csv("error_logs.csv")

# Also parse timestamps to UTC datetimes in a separate column (keep original numeric timestamps as-is)
df_metric['ts_utc'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)
df_trace['ts_utc'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True)
df_log['ts_utc'] = pd.to_datetime(df_log['timestamp'], unit='s', utc=True)
df_error['ts_utc'] = pd.to_datetime(df_error['timestamp'], unit='s', utc=True)

# 1) metric.csv aggregates: distinct (cmdb_id, kpi_name) pairs with counts, top 20
metric_pairs = (
    df_metric
    .groupby(['cmdb_id', 'kpi_name'], dropna=False)
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
    .head(20)
)

metric_total_rows = df_metric.shape[0]
metric_distinct_cmdb = df_metric['cmdb_id'].nunique()
metric_distinct_kpi = df_metric['kpi_name'].nunique()
metric_summary = pd.DataFrame({
    'total_metric_rows': [metric_total_rows],
    'distinct_cmdb': [metric_distinct_cmdb],
    'distinct_kpi_name': [metric_distinct_kpi]
})

# 2) trace.csv aggregates: distinct (cmdb_id, trace_name) pairs with counts, top 20
trace_pairs = (
    df_trace
    .groupby(['cmdb_id', 'trace_name'], dropna=False)
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
    .head(20)
)

trace_total_rows = df_trace.shape[0]
trace_distinct_names = df_trace['trace_name'].nunique()
trace_summary = pd.DataFrame({
    'total_trace_rows': [trace_total_rows],
    'distinct_trace_name': [trace_distinct_names]
})

# 3) log.csv aggregates: distinct (cmdb_id, log_name) pairs with counts, top 20
log_pairs = (
    df_log
    .groupby(['cmdb_id', 'log_name'], dropna=False)
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
    .head(20)
)

log_total_rows = df_log.shape[0]
log_distinct_names = df_log['log_name'].nunique()
log_summary = pd.DataFrame({
    'total_log_rows': [log_total_rows],
    'distinct_log_name': [log_distinct_names]
})

# 4) error_logs.csv: total number of rows and up to first 5 rows (timestamp, cmdb_id, message) as samples
error_total_rows = df_error.shape[0]
error_samples = df_error.loc[:, ['timestamp', 'cmdb_id', 'message']].head(5)

# Display compact results (variables separated by commas so IPython will show them)
metric_pairs, metric_summary, trace_pairs, trace_summary, log_pairs, log_summary, error_total_rows, error_samples

```
Out[1]:
```
summary = (
    "Summary of telemetry files:\n"
    "- metric.csv: 1,900 rows, 15 distinct cmdb_id, 8 distinct KPI names. Top (cmdb_id, kpi_name) pairs (examples) include carts:(cpu, latency-50, latency-90, mem, socket, workload), "
    "carts-db:(cpu, diskio, mem, socket), catalogue and catalogue-db with similar KPI coverage. Top pairs each have 25 rows.\n"
    "- trace.csv: empty (0 rows, 0 distinct trace_name).\n"
    "- log.csv: 450 rows, 2 distinct log_name values (log.error_count, log.row_count) across many services (front-end, catalogue, queue-master, shipping, orders, payment, user, carts, carts-db, session-db, etc.). Top pairs shown have counts up to 25.\n"
    "- error_logs.csv: 0 rows (no error message samples).\n\n"
    "Conclusion: Metric and aggregated log counters are available for many services and can be used for analysis. Trace telemetry and raw error log messages are missing, which limits detailed tracing or message-level root-cause analysis."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(         cmdb_id    kpi_name  count
0          carts         cpu     25
1          carts  latency-50     25
2          carts  latency-90     25
3          carts         mem     25
4          carts      socket     25
5          carts    workload     25
6       carts-db         cpu     25
7       carts-db      diskio     25
8       carts-db         mem     25
9       carts-db      socket     25
10     catalogue         cpu     25
11     catalogue  latency-50     25
12     catalogue  latency-90     25
13     catalogue         mem     25
14     catalogue      socket     25
15     catalogue    workload     25
16  catalogue-db         cpu     25
17  catalogue-db      diskio     25
18  catalogue-db         mem     25
19  catalogue-db      socket     25,    total_metric_rows  distinct_cmdb  distinct_kpi_name
0               1900             15                  8, Empty DataFrame
Columns: [cmdb_id, trace_name, count]
Index: [],    total_trace_rows  distinct_trace_name
0                 0                    0,          cmdb_id         log_name  count
7      front-end    log.row_count     25
6      front-end  log.error_count     25
5      catalogue    log.row_count     25
4      catalogue  log.error_count     25
12  queue-master  log.error_count     25
13  queue-master    log.row_count     25
17      shipping    log.row_count     25
16      shipping  log.error_count     25
8         orders  log.error_count     25
9         orders    log.row_count     25
10       payment  log.error_count     25
11       payment    log.row_count     25
18          user  log.error_count     25
19          user    log.row_count     25
2       carts-db  log.error_count     24
3       carts-db    log.row_count     24
0          carts  log.error_count     21
1          carts    log.row_count     21
15    session-db    log.row_count      5
14    session-db  log.error_count      5,    total_log_rows  distinct_log_name
0             450                  2, 0, Empty DataFrame
Columns: [timestamp, cmdb_id, message]
Index: [])```
```

In [ ]:
# In[2]:

import numpy as np
import pandas as pd

# Reuse previously loaded DataFrames if available; otherwise load them
try:
    df_metric
except NameError:
    df_metric = pd.read_csv("metric.csv")
    df_metric['ts_utc'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)

try:
    df_log
except NameError:
    df_log = pd.read_csv("log.csv")
    df_log['ts_utc'] = pd.to_datetime(df_log['timestamp'], unit='s', utc=True)

try:
    df_trace
except NameError:
    df_trace = pd.read_csv("trace.csv")
    df_trace['ts_utc'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True)

try:
    df_error
except NameError:
    df_error = pd.read_csv("error_logs.csv")
    df_error['ts_utc'] = pd.to_datetime(df_error['timestamp'], unit='s', utc=True)

# Ensure ts_utc exists for metric and log (created above if missing)
if 'ts_utc' not in df_metric.columns:
    df_metric['ts_utc'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)
if 'ts_utc' not in df_log.columns:
    df_log['ts_utc'] = pd.to_datetime(df_log['timestamp'], unit='s', utc=True)

# Incident window (UTC)
win_start = pd.to_datetime("2024-01-20 14:59:52", utc=True)
win_end   = pd.to_datetime("2024-01-20 15:29:52", utc=True)

# 1) metric.csv: compute global P95 per (cmdb_id, kpi_name)
p95_metric = (
    df_metric
    .groupby(['cmdb_id', 'kpi_name'], dropna=False)['value']
    .quantile(0.95)
    .reset_index(name='global_P95')
)

# Merge global P95 into full metric series, then filter to incident window
df_metric_with_p95 = df_metric.merge(p95_metric, on=['cmdb_id', 'kpi_name'], how='left')
df_metric_window = df_metric_with_p95[(df_metric_with_p95['ts_utc'] >= win_start) & (df_metric_with_p95['ts_utc'] <= win_end)].copy()

# Aggregate per pair within window
def agg_metric_group(g):
    cnt = g.shape[0]
    if cnt == 0:
        return pd.Series({
            'count_points_in_window': 0,
            'anomaly_count_in_window': 0,
            'earliest_anomaly_timestamp': np.nan,
            'max_value_in_window': np.nan
        })
    gp95 = g['global_P95'].iloc[0]
    anomalies = g[g['value'] >= gp95]
    anomaly_count = anomalies.shape[0]
    earliest_ts = anomalies['timestamp'].min() if anomaly_count > 0 else np.nan
    max_val = g['value'].max()
    return pd.Series({
        'count_points_in_window': cnt,
        'anomaly_count_in_window': anomaly_count,
        'earliest_anomaly_timestamp': earliest_ts,
        'max_value_in_window': max_val
    })

metric_agg_window = (
    df_metric_window
    .groupby(['cmdb_id', 'kpi_name'], dropna=False)
    .apply(agg_metric_group)
    .reset_index()
)

# Join with global P95 (include pairs that may have zero points in window by starting from p95_metric)
metric_summary = p95_metric.merge(metric_agg_window, on=['cmdb_id','kpi_name'], how='left')

# Fill NaNs for groups with no points in window
metric_summary['count_points_in_window'] = metric_summary['count_points_in_window'].fillna(0).astype(int)
metric_summary['anomaly_count_in_window'] = metric_summary['anomaly_count_in_window'].fillna(0).astype(int)
# earliest_anomaly_timestamp and max_value_in_window remain NaN where appropriate

# Compute ratio (handle global_P95 == 0)
def compute_ratio(row):
    gp95 = row['global_P95']
    maxv = row['max_value_in_window']
    if pd.isna(maxv):
        return np.nan
    if gp95 == 0:
        return np.inf
    return maxv / gp95

metric_summary['ratio'] = metric_summary.apply(compute_ratio, axis=1)

# Sort and limit to top 20
metric_anomalies = (
    metric_summary
    .sort_values(['anomaly_count_in_window', 'ratio'], ascending=[False, False])
    .loc[:, ['cmdb_id','kpi_name','global_P95','count_points_in_window','anomaly_count_in_window','earliest_anomaly_timestamp','max_value_in_window','ratio']]
    .head(20)
)

# 2) log.csv: same procedure for (cmdb_id, log_name)
p95_log = (
    df_log
    .groupby(['cmdb_id', 'log_name'], dropna=False)['value']
    .quantile(0.95)
    .reset_index(name='global_P95')
)

df_log_with_p95 = df_log.merge(p95_log, on=['cmdb_id','log_name'], how='left')
df_log_window = df_log_with_p95[(df_log_with_p95['ts_utc'] >= win_start) & (df_log_with_p95['ts_utc'] <= win_end)].copy()

def agg_log_group(g):
    cnt = g.shape[0]
    if cnt == 0:
        return pd.Series({
            'count_points_in_window': 0,
            'anomaly_count_in_window': 0,
            'earliest_anomaly_timestamp': np.nan,
            'max_value_in_window': np.nan
        })
    gp95 = g['global_P95'].iloc[0]
    anomalies = g[g['value'] >= gp95]
    anomaly_count = anomalies.shape[0]
    earliest_ts = anomalies['timestamp'].min() if anomaly_count > 0 else np.nan
    max_val = g['value'].max()
    return pd.Series({
        'count_points_in_window': cnt,
        'anomaly_count_in_window': anomaly_count,
        'earliest_anomaly_timestamp': earliest_ts,
        'max_value_in_window': max_val
    })

log_agg_window = (
    df_log_window
    .groupby(['cmdb_id', 'log_name'], dropna=False)
    .apply(agg_log_group)
    .reset_index()
)

log_summary = p95_log.merge(log_agg_window, on=['cmdb_id','log_name'], how='left')
log_summary['count_points_in_window'] = log_summary['count_points_in_window'].fillna(0).astype(int)
log_summary['anomaly_count_in_window'] = log_summary['anomaly_count_in_window'].fillna(0).astype(int)
log_summary['ratio'] = log_summary.apply(lambda r: (np.inf if r['global_P95']==0 and not pd.isna(r['max_value_in_window']) else (r['max_value_in_window']/r['global_P95'] if not pd.isna(r['max_value_in_window']) and r['global_P95']!=0 else np.nan)), axis=1)

log_anomalies = (
    log_summary
    .sort_values(['anomaly_count_in_window','ratio'], ascending=[False, False])
    .loc[:, ['cmdb_id','log_name','global_P95','count_points_in_window','anomaly_count_in_window','earliest_anomaly_timestamp','max_value_in_window','ratio']]
    .head(20)
)

# 3) Confirm whether trace.csv and error_logs.csv are empty by returning their total row counts
trace_error_counts = pd.DataFrame({
    'file': ['trace.csv','error_logs.csv'],
    'total_rows': [df_trace.shape[0], df_error.shape[0]]
})

# Display the three compact tables
metric_anomalies, log_anomalies, trace_error_counts

```
Out[2]:
```
Summary of findings (plain English):

- Overall: trace.csv and error_logs.csv are empty (0 rows each), so no trace or raw error-message data is available.

Metric anomalies (top findings):
- Many (cmdb_id, kpi) pairs show anomaly_count_in_window = 25 (every point in the 30‑min window flagged). Several of these have global_P95 = 0, which produces ratio = Inf (catalogue-db:diskio, front-end:error, payment:diskio, rabbitmq:diskio, user:error, etc.). Note: a global P95 of zero makes the “>= P95” rule trigger or produce Inf ratio and can create misleading alerts.
- Several socket KPIs (e.g., carts-db:socket, catalogue:socket, catalogue-db:socket, orders-db:socket, queue-master:socket, rabbitmq:socket, session-db:socket, user:socket, user-db:socket) have anomaly_count_in_window = 25 and ratio = 1.0 — meaning the window max equals the historical P95 (no spike above baseline).
- More actionable spikes (ratio > 1 and nonzero P95) include:
  - rabbitmq-exporter:cpu — anomaly_count 2, ratio ≈ 1.83 (max about 2.38e-4 vs global_P95 ≈ 1.30e-4).
  - queue-master:diskio — anomaly_count 2, ratio ≈ 1.475.
  - queue-master:cpu — anomaly_count 2, ratio ≈ 1.17.
  - shipping:workload — anomaly_count 2, ratio ≈ 1.16.
- Earliest anomaly timestamps (epoch seconds) are present per pair in the table for reference.

Log anomalies (top findings):
- Many services have global_P95 = 0 for log.error_count and anomaly_count equal the window count (25 or fewer), with max = 0 and ratio = Inf (catalogue, front-end, orders, payment, queue-master, shipping, user, carts-db, carts, session-db). This is again a baseline-zero artifact.
- For log.row_count, some services show small spikes above P95:
  - carts: log.row_count — global_P95 125, count_in_window 21, anomaly_count 3, max 150, ratio 1.20.
  - orders, queue-master, shipping, payment, user, catalogue, front-end also show a few anomalies (anomaly_count 2–3) with small ratios (~1.02–1.05).
  - session-db: log.row_count — global_P95 5, anomaly_count 5, max 5, ratio 1.0 (equal to P95).
- Earliest anomaly timestamps for logs are present in the table (epoch seconds).

Interpretation / recommendation:
- Trace and error-log telemetry are missing, limiting detailed root-cause tracing.
- Many metric/log “anomalies” are driven by global P95 == 0; treat those Infinity ratios with caution (they often indicate a zero baseline rather than a true spike).
- Actionable candidates to investigate first: queue-master (diskio and cpu), rabbitmq-exporter (cpu), shipping (workload), and services with clear row_count increases (carts, orders, queue-master, shipping, payment). These show nonzero P95 and ratios >1 or clear increases above baseline.
- Socket metrics mostly equal their P95 (ratio≈1.0) — they do not indicate clear spikes above baseline in this window.

The original code execution output of IPython Kernel is also provided below for reference:

(              cmdb_id  kpi_name    global_P95  count_points_in_window  anomaly_count_in_window  earliest_anomaly_timestamp  max_value_in_window     ratio
17       catalogue-db    diskio  0.000000e+00                      25                       25                1.705763e+09         0.000000e+00       inf
21          front-end     error  0.000000e+00                      25                       25                1.705763e+09         0.000000e+00       inf
38            payment    diskio  0.000000e+00                      25                       25                1.705763e+09         1.078749e+05       inf
49           rabbitmq    diskio  0.000000e+00                      25                       25                1.705763e+09         0.000000e+00       inf
66               user     error  0.000000e+00                      25                       25                1.705763e+09         0.000000e+00       inf
9            carts-db    socket  7.000000e+00                      25                       25                1.705763e+09         7.000000e+00  1.000000
14          catalogue    socket  6.000000e+00                      25                       25                1.705763e+09         6.000000e+00  1.000000
19       catalogue-db    socket  4.000000e+00                      25                       25                1.705763e+09         4.000000e+00  1.000000
36          orders-db    socket  5.000000e+00                      25                       25                1.705763e+09         5.000000e+00  1.000000
47       queue-master    socket  3.000000e+00                      25                       25                1.705763e+09         3.000000e+00  1.000000
51           rabbitmq    socket  1.100000e+01                      25                       25                1.705763e+09         1.100000e+01  1.000000
53  rabbitmq-exporter       mem  4.661248e+06                      25                       25                1.705763e+09         4.661248e+06  1.000000
54  rabbitmq-exporter    socket  1.000000e+00                      25                       25                1.705763e+09         1.000000e+00  1.000000
58         session-db    socket  3.000000e+00                      25                       25                1.705763e+09         3.000000e+00  1.000000
70               user    socket  1.700000e+01                      25                       25                1.705763e+09         1.700000e+01  1.000000
75            user-db    socket  8.000000e+00                      25                       25                1.705763e+09         8.000000e+00  1.000000
52  rabbitmq-exporter       cpu  1.304155e-04                      25                        2                1.705764e+09         2.381319e-04  1.825948
45       queue-master    diskio  5.289545e+04                      25                        2                1.705763e+09         7.802607e+04  1.475100
44       queue-master       cpu  7.068731e-01                      25                        2                1.705764e+09         8.286879e-01  1.172329
64           shipping  workload  2.225033e+00                      25                        2                1.705763e+09         2.574875e+00  1.157230,          cmdb_id         log_name  global_P95  count_points_in_window  anomaly_count_in_window  earliest_anomaly_timestamp  max_value_in_window     ratio
4      catalogue  log.error_count         0.0                      25                       25                  1705762920                    0       inf
6      front-end  log.error_count         0.0                      25                       25                  1705762920                    0       inf
8         orders  log.error_count         0.0                      25                       25                  1705762920                    0       inf
10       payment  log.error_count         0.0                      25                       25                  1705762920                    0       inf
12  queue-master  log.error_count         0.0                      25                       25                  1705762920                    0       inf
16      shipping  log.error_count         0.0                      25                       25                  1705762920                    0       inf
18          user  log.error_count         0.0                      25                       25                  1705762920                    0       inf
2       carts-db  log.error_count         0.0                      24                       24                  1705762980                    0       inf
3       carts-db    log.row_count         1.0                      24                       24                  1705762980                    2  2.000000
0          carts  log.error_count         0.0                      21                       21                  1705762980                    0       inf
14    session-db  log.error_count         0.0                       5                        5                  1705763100                    0       inf
15    session-db    log.row_count         5.0                       5                        5                  1705763100                    5  1.000000
1          carts    log.row_count       125.0                      21                        3                  1705763040                  150  1.200000
9         orders    log.row_count       132.0                      25                        3                  1705763040                  138  1.045455
13  queue-master    log.row_count       594.0                      25                        3                  1705763040                  621  1.045455
17      shipping    log.row_count        66.0                      25                        3                  1705763040                   69  1.045455
11       payment    log.row_count       106.0                      25                        3                  1705763040                  109  1.028302
19          user    log.row_count       631.0                      25                        2                  1705763040                  652  1.033281
5      catalogue    log.row_count       169.6                      25                        2                  1705763040                  174  1.025943
7      front-end    log.row_count      2093.2                      25                        2                  1705763040                 2142  1.023314,              file  total_rows
0       trace.csv           0
1  error_logs.csv           0)```
```

In [ ]:
# In[3]:

import numpy as np
import pandas as pd

# Reuse existing DataFrames (loaded in previous steps)
# Ensure timestamp UTC columns exist
if 'ts_utc' not in df_metric.columns:
    df_metric['ts_utc'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)
if 'ts_utc' not in df_log.columns:
    df_log['ts_utc'] = pd.to_datetime(df_log['timestamp'], unit='s', utc=True)

# Allowed root-cause components
allowed = ["carts","catalogue","orders","payment","user"]

# Incident window (UTC)
win_start = pd.to_datetime("2024-01-20 14:59:52", utc=True)
win_end   = pd.to_datetime("2024-01-20 15:29:52", utc=True)

# 1) Metrics: compute global P95 per (cmdb_id,kpi_name) using full series
p95_metric = (
    df_metric
    .groupby(['cmdb_id', 'kpi_name'], dropna=False)['value']
    .quantile(0.95)
    .reset_index(name='global_P95')
)

# Keep only allowed components
p95_metric_allowed = p95_metric[p95_metric['cmdb_id'].isin(allowed)].copy()

# Merge p95 into full metric series and filter to allowed components and incident window
df_metric_allowed = df_metric[df_metric['cmdb_id'].isin(allowed)].merge(
    p95_metric_allowed, on=['cmdb_id','kpi_name'], how='left'
)
df_metric_window = df_metric_allowed[(df_metric_allowed['ts_utc'] >= win_start) & (df_metric_allowed['ts_utc'] <= win_end)].copy()

# Aggregate per pair within window (only pairs with >0 points in window)
def agg_metric_group(g):
    cnt = g.shape[0]
    gp95 = g['global_P95'].iloc[0]
    anomalies = g[g['value'] >= gp95]
    anomaly_count = anomalies.shape[0]
    earliest_ts = anomalies['timestamp'].min() if anomaly_count > 0 else np.nan
    max_val = g['value'].max()
    ratio = np.inf if (gp95 == 0 and not pd.isna(max_val)) else (max_val / gp95 if not pd.isna(max_val) and gp95 != 0 else np.nan)
    return pd.Series({
        'global_P95': gp95,
        'count_points_in_window': cnt,
        'anomaly_count_in_window': anomaly_count,
        'earliest_anomaly_timestamp': earliest_ts,
        'max_value_in_window': max_val,
        'ratio': ratio
    })

metric_agg_window = (
    df_metric_window
    .groupby(['cmdb_id','kpi_name'], dropna=False)
    .apply(agg_metric_group)
    .reset_index()
)

# Sort and limit to top 20 by anomaly_count desc then ratio desc
metric_allowed_summary = (
    metric_agg_window
    .sort_values(['anomaly_count_in_window','ratio'], ascending=[False, False])
    .loc[:, ['cmdb_id','kpi_name','global_P95','count_points_in_window','anomaly_count_in_window','earliest_anomaly_timestamp','max_value_in_window','ratio']]
    .head(20)
)

# 2) Logs: compute global P95 per (cmdb_id, log_name) using full series, restrict to allowed components
p95_log = (
    df_log
    .groupby(['cmdb_id', 'log_name'], dropna=False)['value']
    .quantile(0.95)
    .reset_index(name='global_P95')
)
p95_log_allowed = p95_log[p95_log['cmdb_id'].isin(allowed)].copy()

df_log_allowed = df_log[df_log['cmdb_id'].isin(allowed)].merge(p95_log_allowed, on=['cmdb_id','log_name'], how='left')
df_log_window = df_log_allowed[(df_log_allowed['ts_utc'] >= win_start) & (df_log_allowed['ts_utc'] <= win_end)].copy()

def agg_log_group(g):
    cnt = g.shape[0]
    gp95 = g['global_P95'].iloc[0]
    anomalies = g[g['value'] >= gp95]
    anomaly_count = anomalies.shape[0]
    earliest_ts = anomalies['timestamp'].min() if anomaly_count > 0 else np.nan
    max_val = g['value'].max()
    ratio = np.inf if (gp95 == 0 and not pd.isna(max_val)) else (max_val / gp95 if not pd.isna(max_val) and gp95 != 0 else np.nan)
    return pd.Series({
        'global_P95': gp95,
        'count_points_in_window': cnt,
        'anomaly_count_in_window': anomaly_count,
        'earliest_anomaly_timestamp': earliest_ts,
        'max_value_in_window': max_val,
        'ratio': ratio
    })

log_agg_window = (
    df_log_window
    .groupby(['cmdb_id','log_name'], dropna=False)
    .apply(agg_log_group)
    .reset_index()
)

log_allowed_summary = (
    log_agg_window
    .sort_values(['anomaly_count_in_window','ratio'], ascending=[False, False])
    .loc[:, ['cmdb_id','log_name','global_P95','count_points_in_window','anomaly_count_in_window','earliest_anomaly_timestamp','max_value_in_window','ratio']]
    .head(20)
)

# Display compact results (variables separated by commas for IPython)
metric_allowed_summary, log_allowed_summary

```
Out[3]:
```
Summary for allowed components (carts, catalogue, orders, payment, user). Global P95 values were computed from the full metric/log series; window = 2024-01-20 14:59:52 → 2024-01-20 15:29:52 UTC.

Key findings — metrics
- payment:diskio — global_P95 = 0, count_in_window = 25, anomaly_count = 25, max = 107874.9, ratio = Inf. Large diskio value vs zero P95 — investigate payment disk I/O (or metric calibration).
- user:error — global_P95 = 0, count_in_window = 25, anomaly_count = 25, max = 0, ratio = Inf. Zero-baseline artifact; treat with caution.
- catalogue:socket and user:socket — anomaly_count = 25, ratio = 1.0 (window max == historical P95). These are at baseline P95, not clear spikes.
- Several KPIs show small, but real, increases above P95 (anomaly_count = 2 each, moderate ratios):
  - user:workload ratio ≈ 1.157
  - payment:workload ratio ≈ 1.140
  - orders:workload ratio ≈ 1.131
  - carts:workload ratio ≈ 1.123
  - carts:cpu ratio ≈ 1.089
  - orders latency/cpu KPIs and various payment KPIs with ratios ≈1.01–1.08
  These indicate modest spikes during the incident window.

Key findings — logs
- log.error_count for carts, catalogue, orders, payment, user: global_P95 = 0 and anomaly_count equals points in window (many 25s). This is a zero-baseline artifact (max = 0, ratio = Inf) and not necessarily a meaningful error-rate spike.
- log.row_count shows small real increases for some services:
  - carts.log.row_count — global_P95 125, anomaly_count 3, max 150, ratio = 1.20 (largest log-volume spike among allowed components).
  - orders, payment, user, catalogue also show a few row_count anomalies with small ratios (~1.03–1.05).
  These suggest increased request/log volume for carts and some other services during the window.

Other notes
- trace.csv and error_logs.csv are empty (no trace or raw error messages to analyze).
- Many “Inf” ratios stem from global_P95 == 0; treat those as baseline/metric-calculation artifacts rather than guaranteed spikes.
- Actionable candidates to investigate first (within allowed components): payment disk I/O (large max vs zero P95), carts (log row_count spike and modest CPU/workload increase), and workload increases in user/payment/orders/carts. Follow up with service logs/traces (if available) and check metric instrumentation for zero-P95 KPIs.

The original code execution output of IPython Kernel is also provided below for reference:

(      cmdb_id    kpi_name    global_P95  count_points_in_window  anomaly_count_in_window  earliest_anomaly_timestamp  max_value_in_window     ratio
19    payment      diskio  0.000000e+00                    25.0                     25.0                1.705763e+09         1.078749e+05       inf
26       user       error  0.000000e+00                    25.0                     25.0                1.705763e+09         0.000000e+00       inf
10  catalogue      socket  6.000000e+00                    25.0                     25.0                1.705763e+09         6.000000e+00  1.000000
30       user      socket  1.700000e+01                    25.0                     25.0                1.705763e+09         1.700000e+01  1.000000
31       user    workload  1.980630e+01                    25.0                      2.0                1.705763e+09         2.290800e+01  1.156602
24    payment    workload  2.207547e+00                    25.0                      2.0                1.705763e+09         2.517000e+00  1.140180
17     orders    workload  2.195670e+00                    25.0                      2.0                1.705763e+09         2.483625e+00  1.131147
5       carts    workload  8.826263e+00                    25.0                      2.0                1.705763e+09         9.908000e+00  1.122559
0       carts         cpu  1.641081e+00                    25.0                      2.0                1.705763e+09         1.787013e+00  1.088924
14     orders  latency-90  1.913826e-01                    25.0                      2.0                1.705764e+09         2.064957e-01  1.078968
16     orders      socket  1.558667e+01                    25.0                      2.0                1.705764e+09         1.653333e+01  1.060736
13     orders  latency-50  7.134242e-02                    25.0                      2.0                1.705764e+09         7.564147e-02  1.060259
11  catalogue    workload  4.422447e+00                    25.0                      2.0                1.705763e+09         4.683000e+00  1.058916
20    payment  latency-50  4.148385e-02                    25.0                      2.0                1.705764e+09         4.330668e-02  1.043941
22    payment         mem  1.303309e+07                    25.0                      2.0                1.705764e+09         1.339569e+07  1.027822
23    payment      socket  1.953667e+01                    25.0                      2.0                1.705764e+09         1.996667e+01  1.022010
25       user         cpu  1.071327e+00                    25.0                      2.0                1.705764e+09         1.092576e+00  1.019834
12     orders         cpu  1.675802e+00                    25.0                      2.0                1.705763e+09         1.696906e+00  1.012594
18    payment         cpu  3.946015e+01                    25.0                      2.0                1.705764e+09         3.991779e+01  1.011598
21    payment  latency-90  8.812601e-02                    25.0                      2.0                1.705764e+09         8.904043e-02  1.010376,      cmdb_id         log_name  global_P95  count_points_in_window  anomaly_count_in_window  earliest_anomaly_timestamp  max_value_in_window     ratio
2  catalogue  log.error_count         0.0                    25.0                     25.0                1.705763e+09                  0.0       inf
4     orders  log.error_count         0.0                    25.0                     25.0                1.705763e+09                  0.0       inf
6    payment  log.error_count         0.0                    25.0                     25.0                1.705763e+09                  0.0       inf
8       user  log.error_count         0.0                    25.0                     25.0                1.705763e+09                  0.0       inf
0      carts  log.error_count         0.0                    21.0                     21.0                1.705763e+09                  0.0       inf
1      carts    log.row_count       125.0                    21.0                      3.0                1.705763e+09                150.0  1.200000
5     orders    log.row_count       132.0                    25.0                      3.0                1.705763e+09                138.0  1.045455
7    payment    log.row_count       106.0                    25.0                      3.0                1.705763e+09                109.0  1.028302
9       user    log.row_count       631.0                    25.0                      2.0                1.705763e+09                652.0  1.033281
3  catalogue    log.row_count       169.6                    25.0                      2.0                1.705763e+09                174.0  1.025943)```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np

# Reuse existing DataFrames if available, otherwise load
try:
    df_metric
except NameError:
    df_metric = pd.read_csv("metric.csv")
try:
    df_log
except NameError:
    df_log = pd.read_csv("log.csv")

# Ensure ts_utc exists for filtering (create if missing)
if 'ts_utc' not in df_metric.columns:
    df_metric['ts_utc'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)
if 'ts_utc' not in df_log.columns:
    df_log['ts_utc'] = pd.to_datetime(df_log['timestamp'], unit='s', utc=True)

# Incident window (UTC)
win_start = pd.to_datetime("2024-01-20 14:59:52", utc=True)
win_end   = pd.to_datetime("2024-01-20 15:29:52", utc=True)

# 1) payment diskio from metric.csv
payment_diskio_mask = (
    (df_metric['cmdb_id'] == 'payment') &
    (df_metric['kpi_name'] == 'diskio') &
    (df_metric['ts_utc'] >= win_start) &
    (df_metric['ts_utc'] <= win_end)
)
payment_diskio_rows = df_metric.loc[payment_diskio_mask, ['timestamp','value']].sort_values('timestamp', ascending=True).head(50).reset_index(drop=True)

payment_diskio_count = payment_diskio_rows.shape[0]
payment_diskio_earliest = payment_diskio_rows['timestamp'].min() if payment_diskio_count>0 else np.nan
payment_diskio_latest = payment_diskio_rows['timestamp'].max() if payment_diskio_count>0 else np.nan
payment_diskio_summary = pd.DataFrame([{
    'series': 'payment:diskio',
    'count_rows': payment_diskio_count,
    'earliest_timestamp': payment_diskio_earliest,
    'latest_timestamp': payment_diskio_latest
}])

# 2) carts workload and cpu from metric.csv
carts_mask = (
    (df_metric['cmdb_id'] == 'carts') &
    (df_metric['kpi_name'].isin(['workload','cpu'])) &
    (df_metric['ts_utc'] >= win_start) &
    (df_metric['ts_utc'] <= win_end)
)
carts_rows = df_metric.loc[carts_mask, ['timestamp','kpi_name','value']].sort_values('timestamp', ascending=True).head(50).reset_index(drop=True)

carts_count = carts_rows.shape[0]
carts_earliest = carts_rows['timestamp'].min() if carts_count>0 else np.nan
carts_latest = carts_rows['timestamp'].max() if carts_count>0 else np.nan
carts_summary = pd.DataFrame([{
    'series': "carts:(workload,cpu)",
    'count_rows': carts_count,
    'earliest_timestamp': carts_earliest,
    'latest_timestamp': carts_latest
}])

# 3) log.row_count for carts and payment from log.csv
log_mask = (
    (df_log['cmdb_id'].isin(['carts','payment'])) &
    (df_log['log_name'] == 'log.row_count') &
    (df_log['ts_utc'] >= win_start) &
    (df_log['ts_utc'] <= win_end)
)
log_rows = df_log.loc[log_mask, ['timestamp','cmdb_id','log_name','value']].sort_values('timestamp', ascending=True).head(50).reset_index(drop=True)

log_count = log_rows.shape[0]
log_earliest = log_rows['timestamp'].min() if log_count>0 else np.nan
log_latest = log_rows['timestamp'].max() if log_count>0 else np.nan
log_summary = pd.DataFrame([{
    'series': "carts|payment:log.row_count",
    'count_rows': log_count,
    'earliest_timestamp': log_earliest,
    'latest_timestamp': log_latest
}])

# Return the three result tables and their summaries (kept compact)
payment_diskio_rows, payment_diskio_summary, carts_rows, carts_summary, log_rows, log_summary

```
Out[4]:
```
Summary (timestamps shown as Unix seconds, within 2024-01-20 14:59:52 → 2024-01-20 15:29:52 UTC):

1) payment : diskio (metric.csv)
- Returned rows: 25
- Earliest timestamp: 1705762920
- Latest timestamp: 1705764360
- Notes: mostly 0 values, with a single large spike at timestamp 1705762980 (value ≈ 107,874.9338).

2) carts : workload and cpu (metric.csv)
- Returned rows: 50 (both KPIs interleaved)
- Earliest timestamp: 1705762920
- Latest timestamp: 1705764360
- Notes / examples: at 1705762920 cpu=1.66537, workload=9.908; at 1705762980 cpu≈1.41830, workload≈7.1056; values generally show workload in the ~7–10 range and cpu ~1.4–1.8 across the window.

3) carts and payment : log.row_count (log.csv)
- Returned rows: 46
- Earliest timestamp: 1705762920
- Latest timestamp: 1705764360
- Notes / examples: 1705762920 payment log.row_count=15; 1705762980 carts=25, payment=90; 1705763040 carts=150 (largest carts value in window), payment=109; later values include carts=50 and payment≈102/103/87.

If you want the full (non-truncated) lists or specific min/max per series computed, I can show head/tail or aggregate stats.

The original code execution output of IPython Kernel is also provided below for reference:

(     timestamp          value
0   1705762920       0.000000
1   1705762980  107874.933787
2   1705763040       0.000000
3   1705763100       0.000000
4   1705763160       0.000000
..         ...            ...
20  1705764120       0.000000
21  1705764180       0.000000
22  1705764240       0.000000
23  1705764300       0.000000
24  1705764360       0.000000

[25 rows x 2 columns],            series  count_rows  earliest_timestamp  latest_timestamp
0  payment:diskio          25          1705762920        1705764360,      timestamp  kpi_name     value
0   1705762920       cpu  1.665370
1   1705762920  workload  9.908000
2   1705762980       cpu  1.418302
3   1705762980  workload  7.105600
4   1705763040       cpu  1.527884
..         ...       ...       ...
45  1705764240  workload  8.268917
46  1705764300       cpu  1.449032
47  1705764300  workload  8.296567
48  1705764360       cpu  1.396338
49  1705764360  workload  8.069113

[50 rows x 3 columns],                  series  count_rows  earliest_timestamp  latest_timestamp
0  carts:(workload,cpu)          50          1705762920        1705764360,      timestamp  cmdb_id       log_name  value
0   1705762920  payment  log.row_count     15
1   1705762980    carts  log.row_count     25
2   1705762980  payment  log.row_count     90
3   1705763040    carts  log.row_count    150
4   1705763040  payment  log.row_count    109
..         ...      ...            ...    ...
41  1705764240  payment  log.row_count    102
42  1705764300    carts  log.row_count     50
43  1705764300  payment  log.row_count    103
44  1705764360    carts  log.row_count     50
45  1705764360  payment  log.row_count     87

[46 rows x 4 columns],                         series  count_rows  earliest_timestamp  latest_timestamp
0  carts|payment:log.row_count          46          1705762920        1705764360)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```